<font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

<font size="5" color='grey'> <b>
M09 - Aufgabe A3 FIXED: YOLO + GrabCut Segmentation → Präzisions-Maske → Comic
</b></font> </br>

---

# Aufgabenbeschreibung

**Hybrid-Workflow mit maximaler Präzision:**

1. **YOLO Object Detection**: Findet Objekte und gibt exakte Koordinaten
2. **GrabCut Segmentation**: Erstellt Pixel-genaue Masken *innerhalb* der YOLO-Koordinaten
3. **Comic-Effekt**: Wende Comic-Filter mit der präzisen Maske an

**Vorteil gegenüber A2**: Pixel-genaue Segmentierung statt nur Rechteck-Masken!

---

# 1 | Umgebung einrichten

In [ ]:
#@title 🔧 Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mprint, install_packages
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
    'ultralytics>=8.0.0',
    'opencv-python>=4.8.0',
    'pillow>=10.0.0'
])

In [ ]:
#@title 📂 Testbilder herunterladen { display-mode: "form" }
!rm -rf files
!mkdir -p files

!curl -L https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/02_bild/peoples.png -o files/peoples.png
!curl -L https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/02_bild/apfel.png -o files/apfel.png

print("✅ Testbilder heruntergeladen")

# 2 | Imports & Setup

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import os

print("✅ Imports erfolgreich")

# 3 | YOLO + GrabCut Initialization

In [ ]:
print("📥 Lade YOLO v8 Modell...")
yolo_model = YOLO('yolov8m.pt')
print("✅ YOLO geladen")
print("✅ OpenCV GrabCut Segmentierung bereit (robust & zuverlässig)")

# 4 | YOLO Object Detection

In [ ]:
def detect_objects_yolo(image_path, conf_threshold=0.5):
    """
    YOLO Objekterkennung mit Koordinaten.
    """
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    results = yolo_model.predict(source=image_path, conf=conf_threshold, verbose=False)
    
    detections = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = box.conf.cpu().numpy()[0]
            cls = int(box.cls.cpu().numpy()[0])
            class_name = yolo_model.names[cls]
            
            detections.append({
                'class_name': class_name,
                'confidence': conf,
                'bbox': (x1, y1, x2, y2),
                'center': ((x1 + x2) // 2, (y1 + y2) // 2)
            })
    
    return image_rgb, detections, results

test_image_path = "files/peoples.png"
image_rgb, detections, _ = detect_objects_yolo(test_image_path)

print(f"✅ {len(detections)} Objekte erkannt")
for i, det in enumerate(detections[:3], 1):
    print(f"  {i}. {det['class_name']} (Conf: {det['confidence']:.2f})")

# 5 | Hybrid Mask Creation: YOLO + GrabCut (ROBUST)

In [ ]:
def create_mask_with_grabcut(image_rgb, bbox, object_class, use_grabcut=True):
    """
    Erstellt Maske mit OpenCV GrabCut für Vordergrund-Segmentierung.
    Funktioniert robust für alle Objekt-Typen!
    """
    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = bbox
    
    # Sicherstellen dass Bounding Box gültig ist
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)
    
    if use_grabcut:
        print(f"  🎯 Verwende GrabCut für {object_class} Segmentierung...")
        try:
            # Kopie des Bildes für GrabCut
            image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
            mask = np.zeros((h, w), dtype=np.uint8)
            
            # GrabCut initialisieren mit Rechteck
            bgdModel = np.zeros((1, 65), np.float64)
            fgdModel = np.zeros((1, 65), np.float64)
            
            # GrabCut anwenden (5 Iterationen für bessere Segmentierung)
            cv2.grabCut(image_bgr, mask, (x1, y1, x2-x1, y2-y1), bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)
            
            # Mask konvertieren: 0 und 2 sind Background, 1 und 3 sind Foreground
            segmented_mask = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
            
            # Nur auf Bounding Box Bereich anwenden
            final_mask = np.zeros((h, w), dtype=np.uint8)
            final_mask[y1:y2, x1:x2] = segmented_mask[y1:y2, x1:x2]
            
            print("    ✅ GrabCut erfolgreich angewendet")
            return final_mask
            
        except Exception as e:
            print(f"    ⚠️ GrabCut Fehler: {e}. Nutze Fallback...")
            return create_rectangular_mask(h, w, bbox)
    else:
        return create_rectangular_mask(h, w, bbox)

def create_rectangular_mask(h, w, bbox):
    """
    Fallback: Rechteckige Maske aus Bounding Box.
    """
    mask = np.zeros((h, w), dtype=np.uint8)
    x1, y1, x2, y2 = bbox
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)
    mask[y1:y2, x1:x2] = 255
    return mask

# Test mit bester Detection (Person)
best_detection = max(detections, key=lambda x: x['confidence'])
print(f"\n🎯 Beste Detection: {best_detection['class_name']}")
print(f"   Koordinaten: {best_detection['bbox']}")

# Maske mit GrabCut erstellen
mask = create_mask_with_grabcut(
    image_rgb, 
    best_detection['bbox'],
    best_detection['class_name'],
    use_grabcut=True
)

# Morphologische Operationen für glatte Kanten
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.GaussianBlur(mask, (7, 7), 0)

print("✅ Hybrid-Maske (GrabCut) erstellt")

# 6 | Maske Visualisierung (Vergleich)

In [ ]:
# Vergleich: Rechteck-Maske vs GrabCut-Maske
rectangular_mask = create_rectangular_mask(image_rgb.shape[0], image_rgb.shape[1], best_detection['bbox'])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original
axes[0].imshow(image_rgb)
x1, y1, x2, y2 = best_detection['bbox']
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='r', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title("1. Original mit YOLO BBox", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Rechteck-Maske (A2 Methode)
axes[1].imshow(rectangular_mask, cmap='gray')
axes[1].set_title("2. A2: Rechteck-Maske\n(Einfach)", fontsize=12, fontweight='bold')
axes[1].axis('off')

# GrabCut-Maske (A3 Methode)
axes[2].imshow(mask, cmap='gray')
axes[2].set_title("3. A3: GrabCut-Maske\n(Pixel-genau) ⭐", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle("Vergleich: Mask Creation Methoden", fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/mask_comparison_FIXED.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Masken verglichen und visualisiert")

# 7 | Comic-Effekt mit Präzisions-Maske

In [ ]:
def apply_comic_effect(image_rgb, mask=None, edge_threshold=50, num_colors=5):
    """
    Comic-Effekt mit Maske-Blending.
    """
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # Kanten erkennen
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, edge_threshold, edge_threshold * 2)
    edges = cv2.medianBlur(edges, 5)
    
    # Bilateral Filter (Glättung ohne Kantenverwischung)
    smoothed = cv2.bilateralFilter(image_bgr, 9, 75, 75)
    
    # K-Means Clustering für Farbquantisierung
    z = smoothed.reshape((-1, 3))
    z = np.float32(z)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    ret, labels, center = cv2.kmeans(z, num_colors, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    center = np.uint8(center)
    res = center[labels.flatten()]
    posterized = res.reshape(smoothed.shape)
    
    # Kanten-Overlay
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    edges_bgr = (edges_bgr > 0).astype(np.uint8) * 255
    comic = cv2.bitwise_and(posterized, cv2.bitwise_not(edges_bgr))
    comic = np.where(edges_bgr > 0, 0, comic)
    
    comic_rgb = cv2.cvtColor(comic, cv2.COLOR_BGR2RGB)
    
    # Weiches Blending mit normalisierter Maske
    if mask is not None:
        mask_normalized = (mask / 255.0).astype(np.float32)
        mask_3d = np.stack([mask_normalized, mask_normalized, mask_normalized], axis=2)
        comic_rgb = (comic_rgb * mask_3d + image_rgb * (1 - mask_3d)).astype(np.uint8)
    
    return comic_rgb, edges

# Comic-Effekt anwenden
comic_result, edges = apply_comic_effect(image_rgb, mask)
print("✅ Comic-Effekt angewendet")

# 8 | Finale 4-Panel Visualisierung

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Original mit BBox
axes[0, 0].imshow(image_rgb)
x1, y1, x2, y2 = best_detection['bbox']
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='g', facecolor='none')
axes[0, 0].add_patch(rect)
axes[0, 0].set_title('1. YOLO Detection', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

# Panel 2: GrabCut Maske
axes[0, 1].imshow(mask, cmap='gray')
axes[0, 1].set_title('2. GrabCut Maske\n(Pixel-genau Segmentierung)', fontsize=14, fontweight='bold')
axes[0, 1].axis('off')

# Panel 3: Kanten
axes[1, 0].imshow(edges, cmap='gray')
axes[1, 0].set_title('3. Edge Detection (Canny)', fontsize=14, fontweight='bold')
axes[1, 0].axis('off')

# Panel 4: Comic Effect
axes[1, 1].imshow(comic_result)
axes[1, 1].set_title('4. Comic Effect\n(mit GrabCut Maske)', fontsize=14, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle('A3 FIXED: YOLO + GrabCut → Comic Workflow', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('output/workflow_a3_FIXED_complete.png', dpi=100, bbox_inches='tight')
plt.show()

cv2.imwrite('output/a3_FIXED_comic_effect.png', cv2.cvtColor(comic_result, cv2.COLOR_RGB2BGR))
print("✅ Workflow Complete!")

# 9 | Test mit apfel.png (Fallback Test)

In [ ]:
print("\n" + "="*60)
print("Test mit apfel.png (GrabCut auch für Objekte)")
print("="*60 + "\n")

image_apple, detections_apple, _ = detect_objects_yolo('files/apfel.png')

if detections_apple:
    best_apple = max(detections_apple, key=lambda x: x['confidence'])
    print(f"Detected: {best_apple['class_name']}")
    
    mask_apple = create_mask_with_grabcut(
        image_apple,
        best_apple['bbox'],
        best_apple['class_name'],
        use_grabcut=True
    )
    
    # Morphologische Operationen
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask_apple = cv2.morphologyEx(mask_apple, cv2.MORPH_CLOSE, kernel)
    mask_apple = cv2.morphologyEx(mask_apple, cv2.MORPH_OPEN, kernel)
    mask_apple = cv2.GaussianBlur(mask_apple, (7, 7), 0)
    
    comic_apple, _ = apply_comic_effect(image_apple, mask_apple)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(image_apple)
    x1, y1, x2, y2 = best_apple['bbox']
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='r', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].set_title(f"Detection: {best_apple['class_name']}", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(comic_apple)
    axes[1].set_title("Comic Effect (mit GrabCut)", fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    cv2.imwrite('output/a3_FIXED_apple_comic.png', cv2.cvtColor(comic_apple, cv2.COLOR_RGB2BGR))
    print("✅ Apple Test Complete")

# 10 | Zusammenfassung: A3 FIXED

## 🎯 A3 FIXED: Hybrid YOLO + GrabCut

### ✨ Was wurde repariert

**Problem:** MediaPipe Selfie Segmenter funktionierte nicht zuverlässig

**Lösung:** OpenCV GrabCut statt Selfie Segmenter
- ✅ Robust und zuverlässig
- ✅ Funktioniert für alle Objekt-Typen
- ✅ Pixel-genaue Segmentierung
- ✅ Keine externen Abhängigkeiten

### 📊 Vergleich der Methoden

| Aspekt | A2 | A3 Original | A3 FIXED |
|--------|----|----|-----|
| **Maske-Typ** | Rechteck | MediaPipe (Fehler) | GrabCut ✅ |
| **Qualität** | 🟡 Gut | ❌ Nicht funktionstüchtig | ✅ Ausgezeichnet |
| **Zuverlässigkeit** | ✅ Stabil | ❌ Problematisch | ✅ Robust |
| **Alle Objekte** | ✅ Ja | ⚠️ Nur Person | ✅ Ja |

### 🔑 GrabCut Vorteile

1. **Interaktive Segmentierung**: Nutzt Bounding Box als Hinweis
2. **Iterativ**: 5 Iterationen für besseres Resultat
3. **Robust**: Funktioniert ohne externe ML-Modelle
4. **Universell**: Für alle Objekt-Typen geeignet

### 🎨 Resultat

```
A2 (Rechteck)    A3 FIXED (GrabCut)
┌─────────────┐   ┌─────────────────┐
│ Kantig      │   │ Sanft, natürlich│
│ Abrupt      │   │ Präzise         │
│ Einfach     │   │ Intelligent     │
└─────────────┘   └─────────────────┘
```